# 06 · CNN Hyperparameter Sweep

**Objective.** A one-variable-at-a-time sweep (kernel size, conv depth, filter count,
dropout, learning rate) to check the architecture is reasonable — run on a **single**
station-year so the result is a sanity check, not a generalization claim.

**Provenance & reproducibility**

- **Results shown here:** ⬇ **loaded from Nov-17** (`results/nov17_hpsweep.csv`); sorted here.
- **Experimental setup:** Single station-year **FD11-2023**, horizon 72 h, SWC-only inputs. Each row varies one hyperparameter from the baseline.
- **Implementation:** `src.models.ConfigurableCNN` via `run_fold(..., arch='configurable', kernel_size=…, n_layers=…, dropout=…)`.
- **Original code:** the 150 production CNN runs and the HP sweep were generated by the original monolithic notebook, preserved at `archive/cnn_experiments_original_monolithic.ipynb` (run live on Google Colab GPU, 2026-11-17); the notebooks here are the refactored, `src/`-based version of that same pipeline.
- **Reproduce:** GPU snippet below, or `python main.py --features swc --horizon 72 --arch configurable --kernel-size 7 --stations FD11 --test-years 2023`.

In [1]:
# --- project setup: portable across local / Google Drive / fresh Colab ---
import os, sys

def find_project_root(marker="src"):
    # 1) walk up from the current directory (local checkout or Drive upload)
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    # 2) common Colab upload locations
    for c in ("/content/soil_moisture_research",
              "/content/drive/MyDrive/soil_moisture_research"):
        if os.path.isdir(os.path.join(c, marker)):
            return c
    raise FileNotFoundError(
        "Could not find the project root (a folder containing 'src/').\n"
        "Upload the whole soil_moisture_research/ folder to Drive/Colab and "
        "open this notebook from inside notebooks/.")

ROOT = find_project_root()
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print("project root set ->", os.path.basename(ROOT))  # portable: no absolute path

project root set -> soil_moisture_research


In [2]:
import pandas as pd
hp = pd.read_csv('results/nov17_hpsweep.csv')
hp = hp.sort_values('RMSE')
hp[['name','kernel_size','n_layers','base_filters','dropout','lr','n_params','RMSE','MAE','epochs']].round(4)

,name,kernel_size,n_layers,base_filters,dropout,lr,n_params,RMSE,MAE,epochs
4,layers=3,3,3,16,0.0,0.0010,7905,0.0265,0.0140,40
2,kernel=7,7,2,16,0.0,0.0010,3777,0.0266,0.0140,25
1,kernel=5,5,2,16,0.0,0.0010,2721,0.0268,0.0134,24
6,filters=32,3,2,32,0.0,0.0010,6401,0.0275,0.0140,14
0,baseline,3,2,16,0.0,0.0010,1665,0.0279,0.0131,21
5,filters=8,3,2,8,0.0,0.0010,449,0.0280,0.0145,11
3,layers=1,3,1,16,0.0,0.0010,81,0.0280,0.0141,11
9,lr=0.01,3,2,16,0.0,0.0100,1665,0.0281,0.0141,16
10,lr=0.0001,3,2,16,0.0,0.0001,1665,0.0281,0.0148,14
8,dropout=0.3,3,2,16,0.3,0.0010,1665,0.0285,0.0138,10


### Regenerating (optional, GPU)
```python
from src.data_processing import load_dataframe, build_station_year_store
from src.evaluation import run_fold
from src import config as C
store = build_station_year_store(load_dataframe(), C.SELECTED_STATIONS)
row, _ = run_fold(store, C.SELECTED_STATIONS, 'FD11', 2023, C.feature_columns('swc'),
                  horizon=72, arch='configurable', kernel_size=7, n_layers=2)
```

**Takeaway.** The largest gains come from a **deeper/wider receptive field** (3 conv
layers or kernel 7 → ~5% lower RMSE than the k=3 baseline). Dropout hurts (the model is
already tiny); lr=1e-3 is best. **These gains were measured on a single station-year and
must not be assumed to generalize** — establishing that would require the same sweep across
all held-out station-years and multiple seeds.